In [9]:
import sys
from typing import List, Dict, Any
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.stores import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory

# 1. Initialize local intelligence layers
llm = ChatOllama(model="llama3.2", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# 2. Establish our multi-turn chat history state tracker
session_store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = ChatMessageHistory()
    return session_store[session_id]

print("✓ Cell 1: Core environments and model instances loaded cleanly.")

✓ Cell 1: Core environments and model instances loaded cleanly.


In [14]:
# 1. Initialize FAISS with a dummy document to establish vector dimension sizes
placeholder_doc = [Document(page_content="init", metadata={"source": "init"})]
child_vector_store = FAISS.from_documents(placeholder_doc, embeddings) 

# 2. Prepare our isolated parent key-value storage layer
parent_doc_store = InMemoryStore()

# 3. Establish our multi-tier cutting boundaries
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap = 0) 
child_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap = 20)   

# 4. Create the robust Parent-Child implementation class
class ModernParentChildRetriever(BaseRetriever):
    vectorstore: FAISS
    docstore: InMemoryStore
    child_splitter: RecursiveCharacterTextSplitter
    parent_splitter: RecursiveCharacterTextSplitter

    class Config:
        arbitrary_types_allowed = True

    def add_documents(self, documents: List[Document]):
        for doc in documents:
            parent_chunks = self.parent_splitter.split_documents([doc])
            for p_idx, p_chunk in enumerate(parent_chunks):
                parent_id = f"{doc.metadata.get('source', 'doc')}_p_{p_idx}"
                self.docstore.mset([(parent_id, p_chunk)])
                
                child_chunks = self.child_splitter.split_documents([p_chunk])
                for c_chunk in child_chunks:
                    c_chunk.metadata["parent_id"] = parent_id
                self.vectorstore.add_documents(child_chunks)

    def _get_relevant_documents(self, query: str, *, run_manager=None) -> List[Document]:
        child_results = self.vectorstore.similarity_search(query, k=2)
        parent_ids = list(set(c.metadata["parent_id"] for c in child_results if "parent_id" in c.metadata))
        parent_docs = self.docstore.mget(parent_ids)
        return [doc for doc in parent_docs if doc is not None]

# 5. Instantiate our active retriever
retriever = ModernParentChildRetriever(
    vectorstore=child_vector_store,
    docstore=parent_doc_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

print("✓ Cell 2: Custom Parent-Child storage architecture safely initialized with dimensions mapped.")

✓ Cell 2: Custom Parent-Child storage architecture safely initialized with dimensions mapped.


/tmp/ipykernel_106223/876667990.py:13: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class ModernParentChildRetriever(BaseRetriever):


In [15]:
# System documentation payload entry
kb_data = [
    Document(
        page_content="MeshQuery is our internal high-scale distributed indexing engine built in 2026. "
        "It uses a specialized sharding strategy optimized for high-throughput Kafka topics. "
        "Unlike standard relational databases, MeshQuery caches query indexes using H3 spatial indexing clusters "
        "and routes requests across nodes using an ultra-low latency RSocket protocol layer. "
        "The default port for MeshQuery cluster coordination is 9092.",
        metadata={"source": "meshquery_spec_2026.md"}
    )
]

# Run the ingestion layer mapping process
retriever.add_documents(kb_data)

print("✓ Cell 3: Technical specifications processed. Child records mapped to Parent IDs inside the DB.")

✓ Cell 3: Technical specifications processed. Child records mapped to Parent IDs inside the DB.


In [16]:
# 1. Render our system prompt blueprint with historical placeholders
conversational_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert software architecture assistant. Answer the user's question using ONLY "
        "the provided context block below. If the answer cannot be found explicitly in the context, "
        "say 'I do not have that information.'\n\n"
        "Context:\n{context}"
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

# 2. Build the state preparation block to load memory maps on the fly
def prepare_chain_inputs(input_data: dict):
    session_id = input_data.get("session_id", "default")
    question = input_data.get("question", "")
    
    # Extract historical array turns from our memory store
    history_object = get_session_history(session_id)
    chat_history = history_object.messages
    
    # Trigger our parent-child document lookup sequence
    retrieved_documents = retriever.invoke(question)
    context_string = format_docs(retrieved_documents)
    
    return {
        "context": context_string,
        "chat_history": chat_history,
        "question": question
    }

# 3. Assemble our complete declarative sequence via the LCEL pipe operator
stateful_rag_chain = (
    RunnableLambda(prepare_chain_inputs)
    | conversational_prompt
    | llm
    | StrOutputParser()
)

print("✓ Cell 4: Stateful LCEL execution chain successfully compiled.")

✓ Cell 4: Stateful LCEL execution chain successfully compiled.


In [17]:
def run_chat_turn(session_id: str, user_query: str):
    print(f"\nUser: {user_query}")
    print("AI: ", end="", flush=True)
    
    history_obj = get_session_history(session_id)
    payload = {"session_id": session_id, "question": user_query}
    full_response = ""
    
    # Stream tokens instantly to match real-time UI standards
    for chunk in stateful_rag_chain.stream(payload):
        print(chunk, end="", flush=True)
        full_response += chunk
    print("\n")
    
    # Core tracking fix: Save both strings to memory to maintain the referencing loop
    history_obj.add_user_message(user_query)
    history_obj.add_ai_message(full_response)

# --- TRACE A LIVE MULTI-TURN ENGINEERING CONVERSATION ---
active_session = "abhishek_architecture_session"

# Turn 1: Establish explicit context
run_chat_turn(active_session, "What is MeshQuery?")

# Turn 2: Ambiguous follow-up query requiring historical referencing and parent context
run_chat_turn(active_session, "What port does it use?")


User: What is MeshQuery?
AI: MeshQuery is our internal high-scale distributed indexing engine, built in 2026.


User: What port does it use?
AI: The default port for MeshQuery cluster coordination is 9092.

